In [1]:
from pathlib import Path

from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

BASE_URL = "https://gloryolusola.com/alpha"

async with async_playwright() as p:
    browser = await p.chromium.launch(headless=True)
    page = await browser.new_page()
    await page.goto(BASE_URL, wait_until="domcontentloaded")
    await page.wait_for_function(
        """() => {
            const input = document.querySelector('input[placeholder^="Search"]');
            if (!input) return false;
            const match = input.placeholder.match(/Search (\\d+) entries/);
            return Boolean(
                match
                && Number(match[1]) > 0
                && document.querySelectorAll('main .animate-pulse').length === 0
            );
        }""",
        timeout=30_000,
    )
    html = await page.content()
    await browser.close()

soup = BeautifulSoup(html, "html.parser")

# Each saved article is the direct child link of a row (div.group).
# The other anchors on the page are the sidebar, mobile header, and footer.
articles = soup.select("main div.group > a.no-ext-marker[href]")
lines = []
for link in articles:
    title = link.select_one("span")
    title_text = " ".join(title.get_text(strip=True).split()) if title else ""
    lines.append(f"{link['href']} {title_text}".rstrip())

root = Path.cwd().resolve()
if not (root / "pyproject.toml").exists():
    root = root.parent
path = root / "data" / "alpha_entries.txt"
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text("\n".join(lines) + "\n")
other = len(soup.select("a[href]")) - len(articles)
print(f"{len(lines)} article links written to {path}")
print(f"{other} other links on the page were skipped")

3434 article links written to /Users/kelvinajayi/Documents/Github/Personal/chunking/data/alpha_entries.txt
18 other links on the page were skipped


In [2]:
import json

from alpha_search.search import search_alpha

query = "docker compose"
results = search_alpha(query)

print(f"Found {len(results)} results for query: {query}")
print(json.dumps(results, indent=2))

Found 3 results for query: docker compose
[
  {
    "id": 15,
    "url": "https://dave.recoil.org/docker-has-always-used-microvms/",
    "title": "Docker has always used microVMs",
    "score": 1
  },
  {
    "id": 266,
    "url": "https://cursor.com/blog/real-time-rl-for-composer",
    "title": "Improving Composer through real-time RL",
    "score": 1
  },
  {
    "id": 629,
    "url": "https://cursor.com/blog/composer",
    "title": "Composer: Building a fast frontier model with RL",
    "score": 1
  }
]


In [5]:
from alpha_search.fetch import fetch_top_results

pages = fetch_top_results(results, n=3)

for page in pages:
    print(page["score"], page["title"])
    print(page["url"])
    print(page["text"][:400])
    print(len(page["text"]), "chars")
    print()

1 Docker has always used microVMs
https://dave.recoil.org/docker-has-always-used-microvms/
Docker has always used microVMs (well, since 2016)There's a lot of buzz about "microVMs", where a workload runs
        with a stripped down Linux kernel, on a minimalist VMM such asfirecrackeron top of a hypervisor
        like KVM or Xen. MicroVMs are often contrasted to, and considered more secure than,
        traditional Linux Docker containers.
        What if I told you that Docker Desktop 
7441 chars

1 Improving Composer through real-time RL
https://cursor.com/blog/real-time-rl-for-composer
Blog/researchWe are observing unprecedented growth in the usefulness and adoption of coding models in the real world. In the face of 10–100x increases in inference volume, we consider the question: how can we take these trillions of tokens and extract from them a training signal to improve the model?We call our approach of using real inference tokens for training "real-time RL." We first used this
142